#### Setup

In [ ]:
# Using pip
# %pip install torch==2.1.2 --index-url https://download.pytorch.org/whl/cu121

# Using conda
%conda install -y -c pytorch -c nvidia pytorch=2.1.2 pytorch-cuda=12.1

In [ ]:
# HuggingFace ecosystem
%pip install transformers==4.40.2 datasets accelerate -U

# Utilities
%pip install scikit-learn python-dotenv pyarrow numpy==1.26.4 ipywidgets==7.8.5 -U

##### Import packages, set device and label maps

In [ ]:
import notebook_init  # Set visible GPUs
import torch
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from datasets import Dataset

from utils import load_model, load_tokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

LABEL_ID = {
    "HUMAN_GENERATED": 0,
    "MACHINE_GENERATED": 1
}

LABEL_TEXT = {
    0: "HUMAN_GENERATED",
    1: "MACHINE_GENERATED"
}


DEVICE: cuda


##### Change to root for accessing data directory

In [2]:
import os

current_dir = Path.cwd()

if current_dir.name == "notebooks" and (
    Path.exists(current_dir.parent / Path("configs/fine_tune"))
    and Path.exists(current_dir.parent / Path("data"))
):
    os.chdir(current_dir.parent)
    print(f"Current directory: {Path.cwd()}")
elif current_dir.name == "dt133g-thesis-project":
    print(f"Current directory: {Path.cwd()}")
else:
    print("Ensure configs and data exists before running this notebook..")

Current directory: d:\viola\GitHub\dt133g-thesis-project


### Initialize Experiments

##### Configure run with names and paths to models and data directories

In [ ]:
pretrained_models = {
    "cb": "microsoft/codebert-base",
    "gc": "microsoft/graphcodebert-base",
    "ux": "microsoft/unixcoder-base",
    "ct": "Salesforce/codet5p-770",
    "ds": "deepseek-ai/deepseek-coder-1.3b-base",
}

FULL_MODEL_NAME = pretrained_models["gc"]
DATASET_NAME = "codet_m4"
SUBSET_NAME = "test"

MODEL_PATH = f"data/fine_tune/models/{DATASET_NAME}/{FULL_MODEL_NAME}"
MODEL_NAME = FULL_MODEL_NAME.split("/")[-1].rsplit("-")[0]

MODEL_TYPE = "encoder"
if MODEL_NAME == "codet5p":
    MODEL_TYPE = "seq2seq"
elif MODEL_NAME == "deepseek":
    MODEL_TYPE = "causal"
print("MODEL_TYPE =", MODEL_TYPE)

SAVE_DIR = MODEL_PATH / Path("eval_outputs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

##### Initialize dataset loader

In [ ]:
def load_dataset():
    if SUBSET_NAME == "test":
        print(f"Loading dataset: {DATASET_NAME}/test.parquet")
        return Dataset.from_parquet(
            f"data/_06_generated_splits/{DATASET_NAME}/test.parquet",
            columns=["code", "label"]
        )

    else:
        print(f"Loading dataset: {SUBSET_NAME}/augmented_dataset.parquet")
        return Dataset.from_parquet(
            f"data/transformations/{DATASET_NAME}/{SUBSET_NAME}/augmented_dataset.parquet",
            columns=["code", "label"]
        )


##### Initialize dropout logic

In [ ]:
def init_mc_dropout(model):
    """
    Safely configures specific model architectures for MC Dropout evaluation.
    Converts only the intended structural layers into training mode.
    """
    # Freeze global state
    model.eval()
    
    activated_count = 0
    
    if MODEL_NAME in ["codebert", "graphcodebert", "unixcoder"]:
        for module in model.modules():
            if isinstance(module, torch.nn.Dropout):
                module.train()
                activated_count += 1
                
    elif MODEL_NAME == "codet5p":
        for module in model.modules():
            if isinstance(module, torch.nn.Dropout):
                module.train()
                activated_count += 1
                
    elif MODEL_NAME == "deepseek":
        from transformers.models.llama.modeling_llama import LlamaAttention
        
        for module in model.modules():
            if isinstance(module, LlamaAttention):
                module.train()
                activated_count += 1
                
    print(f"[{MODEL_NAME.upper()}] Confirmed: Activated exactly {activated_count} specific structural layers for MC Dropout.")

##### Initialize computation logic

In [ ]:
NUM_PASSES = 20
P_DROP = 0.1
SCALE_CORRECTION = 1.0 - P_DROP

tokenizer = load_tokenizer(FULL_MODEL_NAME)

if MODEL_TYPE == "causal":
    H_ID = tokenizer.encode("H", add_special_tokens=False)[0]
    M_ID = tokenizer.encode("M", add_special_tokens=False)[0]


def compute_tier_metrics(samples, model):
    """
    Runs T stochastic forward passes using architecture-specific MC Dropout.
    Calculating Mean Probabilities, Predictive Entropy, and Epistemic Variance.
    """
    print(f"\nProcessing {SUBSET_NAME.title()} Data on device: {DEVICE.upper()} | Architecture: {MODEL_TYPE.upper()}, Model: {MODEL_NAME.upper()}")
    flat_records = []

    init_mc_dropout(model)

    for idx, sample in enumerate(tqdm(samples, desc=f"Evaluating {SUBSET_NAME.title()} Data")):
        snippet_id = sample.get("snippet_id", idx)
        true_label = sample.get("label", -1) # 0 for HUMAN, 1 for MACHINE

        # print(f"Performing inference on sample {idx + 1}/{len(samples)}...")
        # Prepare static input data based on model type
        if MODEL_TYPE == "seq2seq": # (CodeT5+)
            prompts = [
                "classify: " + sample["raw_code"]
            ]
            enc = tokenizer(
                prompts,
                return_tensors="pt",
                padding="longest",
                truncation=True
            )
            inputs = {
                k: v.to(DEVICE)
                for k, v in enc.items()
            }
        elif MODEL_TYPE == "causal": # (DeepSeek-Coder)
            prompt = (
                "Classify the following code"
                + "(output either HUMAN_GENERATED or MACHINE_GENERATED):\n\n"
                + sample["raw_code"]
            )
            enc = tokenizer(
                prompt,
                return_tensors="pt",
                padding="longest",
                truncation=True
            )
            inputs = {
                k: v.to(DEVICE)
                for k, v in enc.items()
            }
        else: # (CodeBERT, GraphCodeBERT, UniXcoder)
            enc = tokenizer(
                sample["code"],
                truncation=True,
                padding="max_length",
                max_length=512,
                return_tensors="pt"
            )
            inputs = {
                k: v.to(DEVICE)
                for k, v in enc.items()
            }

        all_pass_probs = []

        # Stochastic Loop
        with torch.no_grad():
            for _ in range(NUM_PASSES):
                if MODEL_TYPE == "seq2seq":
                    losses = []

                    for label_str in LABEL_TEXT.values():
                        label_ids = tokenizer(
                            label_str,
                            return_tensors="pt"
                        )["input_ids"].to(DEVICE)

                        out = model(**inputs, labels=label_ids)
                        logits_scaled = out.logits * SCALE_CORRECTION

                        loss_fct = torch.nn.CrossEntropyLoss(reduction="sum")
                        loss = loss_fct(
                            logits_scaled.view(-1, logits_scaled.size(-1)),
                            label_ids.view(-1)
                        )
                        losses.append(loss.item())
                    
                    logits = -torch.tensor(losses)
                    probs = torch.softmax(logits, dim=-1)
                
                elif MODEL_TYPE == "causal":
                    outputs = model(**inputs)

                    next_token_logits = outputs.logits[0, -1, :] * SCALE_CORRECTION

                    logits = torch.tensor(
                        [next_token_logits[H_ID],
                         next_token_logits[M_ID]]
                    )
                    probs = torch.softmax(logits, dim=-1)
                
                else: # MODEL_TYPE == "encoder"
                    outputs = model(**inputs)

                    logits = outputs.logits[0] * SCALE_CORRECTION
                    probs = torch.softmax(logits, dim=-1)

                all_pass_probs.append(probs.cpu())

        # Stack individual stochastic passes into a single tensor [num_passes, num_classes]
        stacked_probs = torch.stack(all_pass_probs)
        # Calculate ensemble mean probabilities across all MC passes
        mean_probs = stacked_probs.mean(dim=0)
        pred_label = mean_probs.argmax().item()
        # Get confidence for true label if available, otherwise for predicted label
        target_idx = true_label if true_label != -1 else int(pred_label)
        conf_target = mean_probs[target_idx].item()
        # Predictive Entropy (Total uncertainty from the mean distribution)
        eps = 1e-9
        predictive_entropy = -torch.sum(
            mean_probs * torch.log(mean_probs + eps)
        ).item()
        # Epistemic Variance (Model disagreement/variance across individual dropout iterations)
        epistemic_variance = torch.sum(
            stacked_probs.var(dim=0, unbiased=True)
        ).item()

        flat_records.append({
            "model_name": MODEL_NAME,
            "model_type": MODEL_TYPE,
            "tier": SUBSET_NAME,
            "snippet_id": snippet_id,
            "true_label": true_label,
            "pred_label": pred_label,
            "is_correct": int(true_label == pred_label) if true_label != -1 else None,
            "conf_target": conf_target,
            "entropy": predictive_entropy,
            "epistemic_variance": epistemic_variance
        })

    print(f"── {SUBSET_NAME} Successfully Processed!\n")
    return pd.DataFrame(flat_records)


### Run experiment

##### Load model and dataset

In [ ]:
model = load_model(MODEL_PATH, tokenizer, DEVICE)
dataset = load_dataset()

print(dataset)

##### Compute metrics and save results

In [ ]:
OUTPUT_CSV = SAVE_DIR / f"ood_metrics_{FULL_MODEL_NAME}.csv"

result_df = compute_tier_metrics(dataset, model)

print(f"\n=== Summary Metrics for {SUBSET_NAME.title()} ===")
print(f"Total Evaluated:      {len(result_df)}")
if result_df["is_correct"].notna().any():
    print(f"Mean Accuracy Score:  {result_df['is_correct'].mean():.4f}")
print(f"Average Entropy:      {result_df['entropy'].mean():.4f}")
print(f"Average Variance:     {result_df['epistemic_variance'].mean():.4f}")

first_num = 5

columns = result_df[["snippet_id", "true_label", "pred_label", "conf_target", "entropy", "epistemic_variance"]][:first_num].to_string(index=False)
print(f"\n {'-'*len(' '.join(columns))}")
print(f"Metrics of first {first_num} samples:\n{columns}")
print(f"{'-'*len(' '.join(columns))}\n")


if not os.path.exists(OUTPUT_CSV):
    result_df.to_csv(OUTPUT_CSV, index=False)
    print(f"✓ Created new tracking repository: {OUTPUT_CSV}")
else:
    result_df.to_csv(OUTPUT_CSV, mode='a', header=False, index=False)
    print(f"✓ Appended records seamlessly to existing repository: {OUTPUT_CSV}")

torch.cuda.empty_cache()